# So sánh các phương pháp Embedding

Notebook này sử dụng output có sẵn từ Parser và Chunker.

- Giữ cùng corpus và câu hỏi khi thay cách embedding.
- TF-IDF → dense tĩnh → E5 → BGE-M3 dense / sparse / multi-vector → hybrid.
- Tìm kiếm trực tiếp bằng NumPy/SciPy để thấy cách tính điểm; chưa dùng vector database.
- CPU, float32, batch nhỏ. Chạy tuần tự từ trên xuống.

Static model chỉ hỗ trợ tiếng Anh; kết quả tiếng Việt là kiểm tra ngoài phạm vi ngôn ngữ của model.
BGE-M3 dùng cùng backbone cho ba biểu diễn, giúp so sánh cách biểu diễn với ít thay đổi hơn.

**Chuẩn bị:** chọn kernel `.venv`. Dependency thêm nằm trong `requirements-embedding.txt`.
Model static và BGE-M3 được lưu tại `output/model_cache/`; nếu máy chưa có cache,
chạy cell tải model ở cuối notebook trước, sau đó Run All.
E5 dùng model đã có từ notebook 03.

# Setup

In [117]:
import gc
import json
import hashlib
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

In [118]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

STRATEGY = "structure_aware_recursive"
CHUNK_PATH = ROOT / "output/chunks" / f"{STRATEGY}.jsonl"
PARSER_PATH = ROOT / "output/parsed_chapter1.jsonl"
CACHE_PATH = ROOT / "output/model_cache"
OUTPUT_PATH = ROOT / "output/embedding_comparison" / STRATEGY
TOP_K = 5
BATCH_SIZE = 2

In [119]:
# set seed torch
torch.set_num_threads(4)
torch.manual_seed(42)

In [120]:
model_info = {
  "static": {
    "name": "sentence-transformers/static-retrieval-mrl-en-v1",
    "revision": "f60985c706f192d45d218078e49e5a8b6f15283a"
  },
  "bge_m3": {
    "name": "BAAI/bge-m3",
    "revision": "5617a9f61b028005a4858fdac845db406aefb181"
  }
}

# Experiment 0: Input và bằng chứng đúng

## Đọc output Chunker

In [121]:
with CHUNK_PATH.open(encoding="utf-8") as stream:
    rows = [json.loads(line) for line in stream if line.strip()]
chunks = [
    Document(page_content=row["page_content"], metadata=row["metadata"])
    for row in rows
]

## Đọc output Parser

In [122]:
with PARSER_PATH.open(encoding="utf-8") as stream:
    records = [json.loads(line) for line in stream if line.strip()]

records_by_id = {record["id"]: record for record in records}

## Quan sát corpus

In [123]:
chunk_summary = pd.DataFrame([
    {
        "row_index": index,
        "chapter": chunk.metadata["chapter_number"],
        "section": chunk.metadata["section_number"],
        "characters": len(chunk.page_content),
        "source_count": len(chunk.metadata["sources"]),
    }
    for index, chunk in enumerate(chunks)
])
chunk_summary.groupby("chapter").agg(
    chunks=("row_index", "size"), characters=("characters", "sum")
)

,chunks,characters
chapter,,
1,102,69866
2,101,68554
3,38,26451


- Chapter 1 được cắt thành 102 chunks với Structure-aware Chunker
- Chapter 2 được cắt thành 101 chunks
- Chapter 3 (hiện tại) được cắt thành 38 chunks

## Chunk chỉ chứa heading không tham gia tìm kiếm

In [124]:
chunk_summary.loc[chunk_summary["source_count"] == 0]

,row_index,chapter,section,characters,source_count
20,20,1,1.3,57,0
54,54,1,1.6.2,50,0
57,57,1,1.6.3,67,0
66,66,1,1.6.8,50,0
70,70,1,1.6.9,78,0
85,85,1,1.7.4,73,0
91,91,1,1.7.6,76,0
162,162,2,2.5.2,52,0


In [125]:
indices = chunk_summary.loc[
    chunk_summary["source_count"] > 0, "row_index"
].tolist()

# Lấy text của các chunk
texts = [chunks[index].page_content for index in indices]
len(texts)

233

## Kiểm tra metadata truy vết

In [126]:
for index in indices:
    chunk = chunks[index]
    assert chunk.page_content.strip()
    for source in chunk.metadata["sources"]:
        record = records_by_id[source["record_id"]]
        original = record["text"][
            source["source_start_index"]:source["source_end_index"]
        ]
        copied = chunk.page_content[
            source["chunk_start_index"]:source["chunk_end_index"]
        ]
        assert original == copied

- Kiểm tra thử đoạn văn trong chunk có khớp với text trong metadata trở về đoạn văn gốc không

## Bộ câu hỏi nhỏ: 8 chủ đề, mỗi chủ đề có một câu Anh và một câu Việt

Mỗi chủ đề gán một **đoạn bằng chứng cụ thể trong Parser**, không gán ID chunk.
Chunk được tính là khớp nếu `sources` chứa toàn bộ khoảng bằng chứng đó.
Đây là đánh giá truy hồi bằng chứng đã gán, chưa phải gold đầy đủ cho mọi đoạn trả lời hợp lệ.

In [127]:
cases = []

In [128]:
cases.append({
    "topic": "dbms",
    "en": "What is a database management system?",
    "vi": "Hệ quản trị cơ sở dữ liệu là gì?",
    "record_suffix": "p36:b2",
    "evidence": "A database management system (DBMS) is a computerized system that enables users to create and maintain a database."
})

In [129]:
cases.append({
    "topic": "schema",
    "en": "What is a database schema?",
    "vi": "Lược đồ cơ sở dữ liệu là gì?",
    "record_suffix": "p64:b4",
    "evidence": "The description of a database is called the database schema, which is specified during database design and is not expected to change frequently."
})

In [130]:
cases.append({
    "topic": "state",
    "en": "What do we call the data in a database at a particular moment?",
    "vi": "Dữ liệu trong cơ sở dữ liệu tại một thời điểm được gọi là gì?",
    "record_suffix": "p65:b3",
    "evidence": "The data in the database at a particular moment in time is called a database state or snapshot."
})

In [131]:
cases.append({
    "topic": "logical",
    "en": "Which data independence allows structural and constraint changes without affecting application programs?",
    "vi": "Loại độc lập dữ liệu nào cho phép thay đổi cấu trúc và ràng buộc mà không ảnh hưởng chương trình ứng dụng?",
    "record_suffix": "p68:b3",
    "evidence": "logical data independence is harder to achieve because it allows structural and constraint changes without affecting application programs"
})

In [132]:
cases.append({
    "topic": "physical",
    "en": "Why can physical files be reorganized without changing the conceptual schema?",
    "vi": "Vì sao có thể tổ chức lại các tệp vật lý mà không đổi lược đồ khái niệm?",
    "record_suffix": "p68:b2",
    "evidence": "Changes to the internal schema may be needed because some physical files were reorganized—for example, by creating additional access structures—to improve the performance of retrieval or update. If the same data as before remains in the database, we should not have to change the conceptual schema."
})

In [133]:
cases.append({
    "topic": "multivalued",
    "en": "Can an attribute have several values for the same entity?",
    "vi": "Một thuộc tính có thể có nhiều giá trị cho cùng một thực thể không?",
    "record_suffix": "p96:b2",
    "evidence": "In some cases an attribute can have a set of values for the same entity—for instance, a Colors attribute for a car, or a College_degrees attribute for a person."
})

In [134]:
cases.append({
    "topic": "derived",
    "en": "Why can age be calculated from birth date instead of being stored separately?",
    "vi": "Vì sao tuổi có thể tính từ ngày sinh thay vì lưu riêng?",
    "record_suffix": "p96:b3",
    "evidence": "The Age attribute is hence called a derived attribute and is said to be derivable from the Birth_date attribute, which is called a stored attribute."
})

In [135]:
cases.append({
    "topic": "key",
    "en": "Which attribute can uniquely identify each entity?",
    "vi": "Thuộc tính nào có thể xác định duy nhất từng thực thể?",
    "record_suffix": "p98:b4",
    "evidence": "Such an attribute is called a key attribute, and its values can be used to identify each entity uniquely."
})

## Ghép câu hỏi Anh–Việt

- Một case gồm topic, en, vi, record_suffix, evidence

In [136]:
queries = [
    {"topic": case["topic"], "language": language, "text": case[language]}
    for case in cases
    for language in ["en", "vi"]
]
query_texts = [query["text"] for query in queries]
pd.DataFrame(queries)

,topic,language,text
0,dbms,en,What is a database management system?
1,dbms,vi,Hệ quản trị cơ sở dữ liệu là gì?
2,schema,en,What is a database schema?
3,schema,vi,Lược đồ cơ sở dữ liệu là gì?
4,state,en,What do we call the data in a database at a pa...
5,state,vi,Dữ liệu trong cơ sở dữ liệu tại một thời điểm ...
6,logical,en,Which data independence allows structural and ...
7,logical,vi,Loại độc lập dữ liệu nào cho phép thay đổi cấu...
8,physical,en,Why can physical files be reorganized without ...
9,physical,vi,Vì sao có thể tổ chức lại các tệp vật lý mà kh...


## Xác định khoảng bằng chứng trong Parser

In [137]:
for case in cases:
    matching = [
        record for record in records
        if record["id"].endswith(":" + case["record_suffix"])
    ]
    assert len(matching) == 1
    record = matching[0]
    assert record["text"].count(case["evidence"]) == 1
    case["record_id"] = record["id"]
    case["start"] = record["text"].index(case["evidence"])
    case["end"] = case["start"] + len(case["evidence"])

## Kiểm tra chunk chứa đủ bằng chứng

In [138]:
def contains_evidence(chunk, case):
    return any(
        source["record_id"] == case["record_id"]
        and source["source_start_index"] <= case["start"]
        and source["source_end_index"] >= case["end"]
        for source in chunk.metadata["sources"]
    )

In [139]:
gold = np.array([
    [contains_evidence(chunks[index], case) for index in indices]
    for case in cases
    for language in ["en", "vi"]
])
assert gold.any(axis=1).all(), "Có bằng chứng chưa được chunk nào chứa đủ."

In [140]:
pd.DataFrame([
    {
        "topic": case["topic"],
        "record_id": case["record_id"],
        "pdf_page": records_by_id[case["record_id"]]["pdf_page"] + 1,
        "matching_chunks": int(gold[2 * index].sum()),
        "evidence": case["evidence"],
    }
    for index, case in enumerate(cases)
])

,topic,record_id,pdf_page,matching_chunks,evidence
0,dbms,863264e73f7a57ea:p36:b2,37,1,A database management system (DBMS) is a compu...
1,schema,863264e73f7a57ea:p64:b4,65,1,The description of a database is called the da...
2,state,863264e73f7a57ea:p65:b3,66,1,The data in the database at a particular momen...
3,logical,863264e73f7a57ea:p68:b3,69,1,logical data independence is harder to achieve...
4,physical,863264e73f7a57ea:p68:b2,69,1,Changes to the internal schema may be needed b...
5,multivalued,863264e73f7a57ea:p96:b2,97,1,In some cases an attribute can have a set of v...
6,derived,863264e73f7a57ea:p96:b3,97,1,The Age attribute is hence called a derived at...
7,key,863264e73f7a57ea:p98:b4,99,1,"Such an attribute is called a key attribute, a..."


## Hàm xem kết quả top-k

In [141]:
scores = {}
times = []
sizes = []
search_times = {}

In [142]:
def top_results(name, query_index=1):
    order = np.argsort(-scores[name][query_index], kind="stable")
    if name in ["tfidf", "bge_sparse"]:
        order = order[scores[name][query_index, order] > 0]
    return pd.DataFrame([
        {
            "rank": rank, "row_index": indices[position],
            "score": float(scores[name][query_index, position]),
            "evidence_match": bool(gold[query_index, position]),
            "chapter": chunks[indices[position]].metadata["chapter_number"],
            "section": chunks[indices[position]].metadata["section_number"],
            "pdf_pages": sorted({s["pdf_page"] + 1 for s in chunks[indices[position]].metadata["sources"]}),
            "preview": texts[position][:160],
        }
        for rank, position in enumerate(order[:TOP_K], start=1)
    ])

# Experiment 1: TF-IDF — sparse truyền thống

TF-IDF biến chunk thành vector thưa: mỗi chiều là một từ/ngram. Term xuất hiện nhiều
trong chunk nhưng ít phổ biến trên corpus được tăng trọng số; term không xuất hiện có giá trị 0.

$$
TF(t,d)=\operatorname{count}(t,d),\qquad
IDF(t)=\ln\frac{N+1}{DF(t)+1}+1,\qquad
w_{t,d}=TF(t,d)\,IDF(t).
$$

$N$ là số chunk; $DF(t)$ là số chunk chứa term $t$. Đây là công thức mặc định
của scikit-learn trong notebook. Sau chuẩn hóa L2, điểm truy xuất là:

$$
\hat w_d=\frac{w_d}{\|w_d\|_2},\qquad s(q,d)=\hat w_q^\top\hat w_d.
$$

Fit vocabulary/IDF trên corpus, dùng cùng vocabulary cho query. Notebook lấy unigram
và bigram. Cách này dễ giải thích và khớp thuật ngữ, nhưng không tự hiểu từ đồng nghĩa
hay nối nghĩa Anh–Việt; query không khớp feature nào sẽ là vector 0.

Đọc thêm: [Embedding Materials — Chương 1](../docs/materials/EmbeddingMaterials.md).

In [143]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), dtype=np.float32, norm="l2")

In [144]:
start = perf_counter()
tfidf_vectors = tfidf.fit_transform(texts)
tfidf_doc_seconds = perf_counter() - start
tfidf_vectors.shape

(233, 16037)

In [145]:
start = perf_counter()
tfidf_queries = tfidf.transform(query_texts)
tfidf_query_seconds = perf_counter() - start
tfidf_queries.shape

(16, 16037)

In [146]:
pd.DataFrame({
    "query": query_texts,
    "matched_features": tfidf_queries.getnnz(axis=1),
})

,query,matched_features
0,What is a database management system?,9
1,Hệ quản trị cơ sở dữ liệu là gì?,0
2,What is a database schema?,7
3,Lược đồ cơ sở dữ liệu là gì?,0
4,What do we call the data in a database at a pa...,20
5,Dữ liệu trong cơ sở dữ liệu tại một thời điểm ...,0
6,Which data independence allows structural and ...,22
7,Loại độc lập dữ liệu nào cho phép thay đổi cấu...,0
8,Why can physical files be reorganized without ...,15
9,Vì sao có thể tổ chức lại các tệp vật lý mà kh...,0


In [147]:
sample_position = int(np.flatnonzero(gold[0])[0])
sample = tfidf_vectors[sample_position]
pd.DataFrame({
    "term": tfidf.get_feature_names_out()[sample.indices],
    "weight": sample.data,
}).nlargest(10, "weight")

,term,weight
12,the,0.227130
9,database,0.195563
67,constructing,0.149433
129,the data,0.139736
68,manipulating,0.138918
98,system that,0.138918
50,the database,0.138679
52,database to,0.131458
69,sharing,0.125671
5,data,0.115035


In [148]:
start = perf_counter()
scores["tfidf"] = (tfidf_queries @ tfidf_vectors.T).toarray()
search_times["tfidf"] = perf_counter() - start

In [149]:
top_results("tfidf", query_index=0)

,rank,row_index,score,evidence_match,chapter,section,pdf_pages,preview
0,1,195,0.234472,False,2,2.7,[86],2.2. Discuss the main categories of data model...
1,2,97,0.124953,False,1,1.9,[59],1.2. What four main types of actions involve d...
2,3,8,0.119555,True,1,1.1,[37],A database management system (DBMS) is a compu...
3,4,196,0.117397,False,2,2.7,"[86, 87]",architectures?\n2.10. Discuss some types of da...
4,5,7,0.102744,False,1,1.1,"[36, 37]",An example of a large commercial database is A...


In [150]:
times.append({"method": "tfidf", "load_s": 0.0,
              "documents_s": tfidf_doc_seconds, "queries_s": tfidf_query_seconds})
sizes.append({"method": "tfidf", "representation": "CSR sparse",
              "dimensions": tfidf_vectors.shape[1], "nonzero_values": tfidf_vectors.nnz,
              "numeric_mib": sum(x.nbytes for x in [tfidf_vectors.data, tfidf_vectors.indices, tfidf_vectors.indptr]) / 1024**2})

# Experiment 2: Dense tĩnh — StaticEmbedding

Dense tĩnh tra vector token đã học rồi lấy trung bình để biểu diễn cả đoạn.
“Tĩnh” nghĩa là cùng token ID luôn được tra ra cùng vector, dù ngữ cảnh thay đổi.

$$
e_i=E[t_i],\qquad v(d)=\frac{1}{n}\sum_{i=1}^{n}e_i,\qquad
\hat v(d)=\frac{v(d)}{\|v(d)\|_2},\qquad
s(q,d)=\hat v(q)^\top\hat v(d).
$$

$E$ là bảng embedding; $n$ là số token. Phép trung bình tạo một vector kích thước
cố định; không có attention để đổi nghĩa token theo câu. Encode ít phép tính nhưng
khó phân biệt từ đa nghĩa hoặc thứ tự của cùng một tập token.

Notebook dùng model pretrained `static-retrieval-mrl-en-v1`, vector 1024 chiều,
không tự huấn luyện Word2Vec. Model dành cho tiếng Anh; đọc riêng kết quả Anh/Việt.

Đọc thêm: [Embedding Materials — Chương 2](../docs/materials/EmbeddingMaterials.md).

## Load model

- Các lệnh `perf_counter()` dùng để tính thời gian load model, thời gian encode.. để so sánh giữa các model

In [151]:
start = perf_counter()
static_model = SentenceTransformer(str(CACHE_PATH / "static"), device="cpu")
static_load_seconds = perf_counter() - start
static_model

SentenceTransformer(
  (0): StaticEmbedding({})
)

## Mã hóa thành vector

In [152]:
start = perf_counter()
static_vectors = static_model.encode(texts, normalize_embeddings=True,
                                     batch_size=32, show_progress_bar=False)
static_doc_seconds = perf_counter() - start
static_vectors.shape

(233, 1024)

In [153]:
start = perf_counter()
static_queries = static_model.encode(query_texts, normalize_embeddings=True,
                                     batch_size=32, show_progress_bar=False)
static_query_seconds = perf_counter() - start
static_queries.shape

(16, 1024)

## Kiểm tra vector và xem một vài giá trị

In [154]:
assert np.isfinite(static_vectors).all()
assert np.allclose(np.linalg.norm(static_vectors, axis=1), 1.0, atol=1e-5)
pd.Series(static_vectors[sample_position][:8], name="first_values")

0    0.009875
1    0.030523
2    0.030612
3   -0.046244
4   -0.002024
5    0.088725
6   -0.015762
7   -0.021135
Name: first_values, dtype: float32

## Tính điểm query với toàn bộ chunk

In [155]:
start = perf_counter()
scores["static"] = static_queries @ static_vectors.T
search_times["static"] = perf_counter() - start

## In ra top result cho câu hỏi 0

In [156]:
top_results("static", query_index=1)

,rank,row_index,score,evidence_match,chapter,section,pdf_pages,preview
0,1,213,0.057991,False,3,3.2,"[93, 94]",[[SECTION 3.2: A Sample Database Application]]...
1,2,205,0.055434,False,3,3.intro,[91],. Section 3.8 introduces some UML class diagra...
2,3,189,0.053066,False,2,2.6,[84],The hierarchical model represents data as hier...
3,4,0,0.039290,False,1,1.intro,[35],[[SECTION 1.intro: Chapter Introduction]]\nsto...
4,5,158,0.038931,False,2,2.4.3,[77],"Application development environments, such as ..."


## Lưu thời gian

In [157]:
times.append({"method": "static", "load_s": static_load_seconds,
              "documents_s": static_doc_seconds, "queries_s": static_query_seconds})
sizes.append({"method": "static", "representation": "one dense vector",
              "dimensions": static_vectors.shape[1], "nonzero_values": None,
              "numeric_mib": static_vectors.nbytes / 1024**2})

## Giải phóng model trước khi sang model tiếp theo

In [158]:
del static_model  # Xóa biến tham chiếu đến model
gc.collect()      # Chạy bộ thu gom rác, trả về số đối tượng thu được

8605

# Experiment 3: E5 — dense theo ngữ cảnh

E5 dùng Transformer để tạo vector token có xét ngữ cảnh, rồi lấy trung bình thành
một vector cho đoạn. Khác static ở các vector trước pooling: $h_i$ phụ thuộc cả chuỗi,
còn $E[t_i]$ chỉ được tra theo token ID.

$$
(h_1,\ldots,h_n)=Transformer(t_1,\ldots,t_n),\qquad
v(d)=\frac{\sum_i m_i h_i}{\sum_i m_i}.
$$

$m_i=1$ cho token hợp lệ, $0$ cho padding. Sau chuẩn hóa L2:

$$
\hat v(d)=\frac{v(d)}{\|v(d)\|_2},\qquad s(q,d)=\hat v(q)^\top\hat v(d).
$$

Giữ `multilingual-e5-small` đã dùng ở notebook 03: vector 384 chiều, tối đa 512 token.
Document thêm `passage: `, query thêm `query: `, kể cả tiếng Việt. Model pretrained
phục vụ truy xuất đa ngôn ngữ; encode tốn tài nguyên hơn static và một vector vẫn có thể mất chi tiết.
Đếm token trước encode để không âm thầm cắt nội dung. Không huấn luyện lại bằng `gold`.

Đọc thêm: [Embedding Materials — Chương 3](../docs/materials/EmbeddingMaterials.md).

In [159]:
start = perf_counter()
e5_model = SentenceTransformer(
    "intfloat/multilingual-e5-small", device="cpu", local_files_only=True,
    revision="614241f622f53c4eeff9890bdc4f31cfecc418b3",
)
e5_load_seconds = perf_counter() - start

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [160]:
passage_inputs = ["passage: " + text for text in texts]
query_inputs = ["query: " + text for text in query_texts]

## Kiểm tra token trước encode, không âm thầm truncate

In [161]:
e5_tokens = [
    len(e5_model.tokenizer(text, truncation=False)["input_ids"])
    for text in passage_inputs + query_inputs
]
assert max(e5_tokens) <= e5_model.max_seq_length
pd.Series(e5_tokens, name="input_tokens").describe()

count    249.000000
mean     152.080321
std       63.489725
min       11.000000
25%      107.000000
50%      163.000000
75%      201.000000
max      296.000000
Name: input_tokens, dtype: float64

In [162]:
start = perf_counter()
e5_vectors = e5_model.encode(
    passage_inputs, batch_size=BATCH_SIZE, normalize_embeddings=True,
    precision="float32", prompt="", show_progress_bar=False,
)
e5_doc_seconds = perf_counter() - start
e5_vectors.shape

(233, 384)

In [163]:
start = perf_counter()
e5_queries = e5_model.encode(
    query_inputs, batch_size=BATCH_SIZE, normalize_embeddings=True,
    precision="float32", prompt="", show_progress_bar=False,
)
e5_query_seconds = perf_counter() - start
e5_queries.shape

(16, 384)

In [164]:
assert np.isfinite(e5_vectors).all()
assert np.allclose(np.linalg.norm(e5_vectors, axis=1), 1.0, atol=1e-5)
pd.Series(e5_vectors[sample_position][:8], name="first_values")

0   -0.005548
1    0.001801
2   -0.083217
3   -0.059172
4    0.006382
5   -0.054823
6    0.071557
7    0.072903
Name: first_values, dtype: float32

In [165]:
start = perf_counter()
scores["e5"] = e5_queries @ e5_vectors.T
search_times["e5"] = perf_counter() - start

In [166]:
top_results("e5", query_index=1)

,rank,row_index,score,evidence_match,chapter,section,pdf_pages,preview
0,1,8,0.803873,True,1,1.1,[37],A database management system (DBMS) is a compu...
1,2,146,0.798343,False,2,2.4.1,"[73, 74]",Let us consider the top part of Figure 2.3 fir...
2,3,202,0.796472,False,2,2.7,[90],"In this chapter, we follow the traditional app..."
3,4,130,0.796358,False,2,2.3.1,[70],In DBMSs where a clear separation is maintaine...
4,5,40,0.796032,False,1,1.4.1,[46],[[SECTION 1.4.1: Database Administrators]]\nIn...


In [167]:
times.append({"method": "e5", "load_s": e5_load_seconds,
              "documents_s": e5_doc_seconds, "queries_s": e5_query_seconds})
sizes.append({"method": "e5", "representation": "one dense vector",
              "dimensions": e5_vectors.shape[1], "nonzero_values": None,
              "numeric_mib": e5_vectors.nbytes / 1024**2})

In [168]:
del e5_model
gc.collect()

0

# Experiment 4: BGE-M3 — một model, ba biểu diễn

BGE-M3 dùng cùng backbone theo ngữ cảnh để tạo dense, sparse và multi-vector.
Ba nhánh giữ thông tin khác nhau và dùng những hàm scoring khác nhau.

- **Dense:** một vector đại diện đoạn từ token đặc biệt `[CLS]`, chuẩn hóa L2.
- **Sparse:** token ID và trọng số không âm được học; chỉ token ID chung đóng góp điểm.
- **Multi-vector:** giữ nhiều vector token; từng query token chọn document token tốt nhất.

Các điểm chính:

$$
s_{dense}(q,d)=v_q^\top v_d,\qquad
s_{sparse}(q,d)=\sum_{t\in T(q)\cap T(d)}w_t(q)w_t(d).
$$

$$
s_{multi}(q,d)=\frac{1}{n_q}\sum_{i=1}^{n_q}\max_{1\le j\le n_d}u_i^\top v_j.
$$

$T(x)$ là tập token ID được lưu; $u_i,v_j$ là vector token của query và chunk.
Dense tính điểm gọn; sparse dễ xem token đóng góp; multi-vector giữ chi tiết hơn
nhưng cần nhiều bộ nhớ và phép tính. Sparse vẫn cần token chung, nên không tự có
khả năng khớp xuyên ngôn ngữ như dense.

Dùng FlagEmbedding để lấy đúng ba output đã học, không thêm tiền tố E5.
Notebook đặt `max_length=512` dù model hỗ trợ đến 8192 token.
**Thời gian encode BGE bên dưới là tổng cho cả ba output, không phải thời gian riêng từng nhánh.**

Đọc thêm: [Embedding Materials — Chương 4–6](../docs/materials/EmbeddingMaterials.md).

In [169]:
from FlagEmbedding import BGEM3FlagModel

In [170]:
start = perf_counter()
bge_model = BGEM3FlagModel(
    str(CACHE_PATH / "bge_m3"), use_fp16=False, devices="cpu",
)
bge_load_seconds = perf_counter() - start

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [171]:
bge_tokens = [
    len(bge_model.tokenizer(text, truncation=False)["input_ids"])
    for text in texts + query_texts
]
assert max(bge_tokens) <= 512
pd.Series(bge_tokens, name="input_tokens").describe()

count    249.000000
mean     150.016064
std       63.625160
min        8.000000
25%      105.000000
50%      161.000000
75%      199.000000
max      294.000000
Name: input_tokens, dtype: float64

In [172]:
start = perf_counter()
bge_vectors = bge_model.encode(
    texts, batch_size=BATCH_SIZE, max_length=512,
    return_dense=True, return_sparse=True, return_colbert_vecs=True,
)
bge_doc_seconds = perf_counter() - start

Inference Embeddings: 100%|██████████| 117/117 [05:23<00:00,  2.76s/it]


In [173]:
start = perf_counter()
bge_queries = bge_model.encode(
    query_texts, batch_size=BATCH_SIZE, max_length=512,
    return_dense=True, return_sparse=True, return_colbert_vecs=True,
)
bge_query_seconds = perf_counter() - start

Inference Embeddings: 100%|██████████| 8/8 [00:03<00:00,  2.46it/s]


## So sánh hình dạng output của cùng một chunk

In [174]:
pd.Series({
    "dense_shape": bge_vectors["dense_vecs"][sample_position].shape,
    "sparse_nonzero": len(bge_vectors["lexical_weights"][sample_position]),
    "multi_shape": bge_vectors["colbert_vecs"][sample_position].shape,
})

dense_shape           (1024,)
sparse_nonzero            113
multi_shape       (212, 1024)
dtype: object

## Kiểm tra vector trước khi tính điểm

In [175]:
assert bge_vectors["dense_vecs"].shape[0] == len(texts)
assert np.isfinite(bge_vectors["dense_vecs"]).all()
assert np.allclose(np.linalg.norm(bge_vectors["dense_vecs"], axis=1), 1.0, atol=1e-5)
assert all(np.isfinite(list(weights.values())).all()
           for weights in bge_vectors["lexical_weights"])
assert all(np.isfinite(vectors).all()
           for vectors in bge_vectors["colbert_vecs"])

## A. Dense: một vector/chunk, tính cosine

BGE-M3 dùng hidden state của `[CLS]` sau Transformer làm đại diện đoạn,
khác mean pooling của E5:

$$
v(d)=\frac{h_{\mathrm{CLS}}(d)}{\|h_{\mathrm{CLS}}(d)\|_2},\qquad
s_{dense}(q,d)=v(q)^\top v(d).
$$

Lấy `dense_vecs`, vector 1024 chiều. Có một vector cho mỗi chunk, không phải một vector cho mỗi token.

In [176]:
start = perf_counter()
scores["bge_dense"] = bge_queries["dense_vecs"] @ bge_vectors["dense_vecs"].T
search_times["bge_dense"] = perf_counter() - start

In [177]:
top_results("bge_dense", query_index=1)

,rank,row_index,score,evidence_match,chapter,section,pdf_pages,preview
0,1,8,0.668254,True,1,1.1,[37],A database management system (DBMS) is a compu...
1,2,40,0.617207,False,1,1.4.1,[46],[[SECTION 1.4.1: Database Administrators]]\nIn...
2,3,152,0.605221,False,2,2.4.1,[75],It is common to have the client program that a...
3,4,145,0.599687,False,2,2.4.1,[73],[[SECTION 2.4.1: DBMS Component Modules]]\nFig...
4,5,23,0.595701,False,1,1.3.1,"[41, 42]",[[SECTION 1.3.1: Self-Describing Nature of a D...


## B. Sparse neural: trọng số token được học

Model tạo trọng số không âm theo ngữ cảnh; nếu token ID lặp lại thì giữ trọng số lớn nhất:

$$
w_t(d)=\max_{i:t_i=t}a_i(d),\qquad
s_{sparse}(q,d)=\sum_{t\in T(q)\cap T(d)}w_t(q)w_t(d).
$$

$a_i$ là trọng số tại vị trí $i$ do model tạo. BGE-M3 lưu trọng số các token xuất hiện
trong text; đây không phải SPLADE expansion. Lấy `lexical_weights`; chiều không lưu có giá trị 0.
Điểm là tích vô hướng sparse, không mặc định là cosine; không dùng TF-IDF để tạo trọng số này.

In [178]:
weights = bge_vectors["lexical_weights"][sample_position]
pd.DataFrame([
    {"token": bge_model.tokenizer.convert_ids_to_tokens(int(token)), "weight": weight}
    for token, weight in weights.items()
]).nlargest(10, "weight")

,token,weight
6,MS,0.376828
1,▁database,0.316869
3,▁system,0.278865
2,▁management,0.274542
5,DB,0.252729
22,▁DB,0.251603
85,▁Manipul,0.247750
26,▁software,0.243430
74,▁meta,0.239179
59,▁definition,0.222612


In [179]:
def sparse_score(query, document):
    return sum(weight * document.get(token, 0.0)
               for token, weight in query.items())

In [180]:
start = perf_counter()
scores["bge_sparse"] = np.array([
    [sparse_score(query, document) for document in bge_vectors["lexical_weights"]]
    for query in bge_queries["lexical_weights"]
], dtype=np.float32)
search_times["bge_sparse"] = perf_counter() - start

In [181]:
top_results("bge_sparse", query_index=1)

,rank,row_index,score,evidence_match,chapter,section,pdf_pages,preview
0,1,195,0.005281,False,2,2.7,[86],2.2. Discuss the main categories of data model...
1,2,196,0.004097,False,2,2.7,"[86, 87]",architectures?\n2.10. Discuss some types of da...
2,3,97,0.001091,False,1,1.9,[59],1.2. What four main types of actions involve d...


## C. Multi-vector: MaxSim theo token

Mỗi query token chọn document token có điểm cao nhất; BGE-M3 lấy trung bình các max này:

$$
s_{multi}(q,d)=\frac{1}{n_q}\sum_{i=1}^{n_q}\max_{1\le j\le n_d}u_i^\top v_j.
$$

$n_q,n_d$ là số vector token của query và chunk. Lấy `colbert_vecs`, tính ma trận điểm
rồi `max(axis=1).mean()`. Giữ nhiều vector giúp đối chiếu chi tiết nhưng tốn dung lượng
và scoring hơn. Đây là ColBERT-style của BGE-M3, không phải model ColBERTv2 độc lập.

In [182]:
def multi_score(query, document):
    token_scores = query @ document.T
    return float(token_scores.max(axis=1).mean())

In [183]:
token_scores = (
    bge_queries["colbert_vecs"][0]
    @ bge_vectors["colbert_vecs"][sample_position].T
)
pd.DataFrame({
    "query_token_index": np.arange(token_scores.shape[0]),
    "best_document_token_index": token_scores.argmax(axis=1),
    "max_score": token_scores.max(axis=1),
})

,query_token_index,best_document_token_index,max_score
0,0,8,0.808525
1,1,27,0.846977
2,2,0,0.813845
3,3,1,0.879778
4,4,2,0.864545
5,5,3,0.869046
6,6,23,0.761013
7,7,210,0.787591


In [184]:
start = perf_counter()
scores["bge_multi"] = np.array([
    [multi_score(query, document) for document in bge_vectors["colbert_vecs"]]
    for query in bge_queries["colbert_vecs"]
], dtype=np.float32)
search_times["bge_multi"] = perf_counter() - start

In [185]:
top_results("bge_multi", query_index=1)

,rank,row_index,score,evidence_match,chapter,section,pdf_pages,preview
0,1,8,0.655944,True,1,1.1,[37],A database management system (DBMS) is a compu...
1,2,40,0.617804,False,1,1.4.1,[46],[[SECTION 1.4.1: Database Administrators]]\nIn...
2,3,7,0.589383,False,1,1.1,"[36, 37]",An example of a large commercial database is A...
3,4,10,0.585739,False,1,1.1,[37],It is not absolutely necessary to use general-...
4,5,145,0.582153,False,2,2.4.1,[73],[[SECTION 2.4.1: DBMS Component Modules]]\nFig...


## Đối chiếu hai hàm scoring cơ bản với thư viện

In [186]:
assert np.isclose(
    scores["bge_sparse"][0, sample_position],
    bge_model.compute_lexical_matching_score(
        bge_queries["lexical_weights"][0], weights), atol=1e-5,
)
assert np.isclose(
    scores["bge_multi"][0, sample_position],
    float(bge_model.colbert_score(
        bge_queries["colbert_vecs"][0],
        bge_vectors["colbert_vecs"][sample_position])), atol=1e-5,
)

In [187]:
times.append({"method": "bge_all_three", "load_s": bge_load_seconds,
              "documents_s": bge_doc_seconds, "queries_s": bge_query_seconds})

In [188]:
sparse_count = sum(len(weights) for weights in bge_vectors["lexical_weights"])
sizes.extend([
    {"method": "bge_dense", "representation": "one dense vector",
     "dimensions": bge_vectors["dense_vecs"].shape[1], "nonzero_values": None,
     "numeric_mib": bge_vectors["dense_vecs"].nbytes / 1024**2},
    {"method": "bge_sparse", "representation": "token IDs + weights",
     "dimensions": len(bge_model.tokenizer), "nonzero_values": sparse_count,
     "numeric_mib": sparse_count * 8 / 1024**2},
    {"method": "bge_multi", "representation": "token vectors",
     "dimensions": bge_vectors["colbert_vecs"][0].shape[1], "nonzero_values": None,
     "numeric_mib": sum(v.nbytes for v in bge_vectors["colbert_vecs"]) / 1024**2},
])

In [197]:
del bge_model
gc.collect()

0

# Experiment 5: Hybrid — dense + sparse bằng RRF

Hybrid kết hợp danh sách xếp hạng từ BGE dense và BGE sparse, không tạo embedding mới.
RRF dùng thứ hạng thay vì cộng trực tiếp hai thang điểm similarity:

$$
s_{RRF}(q,d)=\sum_{r\in\mathcal R_d}\frac{1}{k+\operatorname{rank}_r(q,d)}.
$$

$\mathcal R_d$ là các nhánh có chunk $d$ trong danh sách ứng viên; hạng bắt đầu từ 1.
Nhánh không có chunk thì đóng góp 0. Notebook giữ $k=60$, không tune trên bộ câu hỏi này.

Chỉ lấy top 20 ứng viên mỗi nhánh; sparse chỉ đóng góp nếu điểm > 0, rồi đánh giá top 5
sau fusion. Các nhánh có thể bổ sung nhau, nhưng nhánh yếu cũng có thể kéo ứng viên sai
lên cao; hybrid không mặc định tốt hơn dense.

Đọc thêm: [Embedding Materials — Chương 7](../docs/materials/EmbeddingMaterials.md).

In [198]:
def rrf(matrix, sparse=False):
    order = np.argsort(-matrix, axis=1, kind="stable")
    ranks = np.argsort(order, axis=1) + 1
    valid = (ranks <= 20) & ((matrix > 0) if sparse else True)
    return np.where(valid, 1.0 / (60 + ranks), 0.0)

In [199]:
start = perf_counter()
scores["hybrid"] = rrf(scores["bge_dense"]) + rrf(scores["bge_sparse"], sparse=True)
search_times["hybrid"] = perf_counter() - start

In [200]:
top_results("hybrid", query_index=1)

,rank,row_index,score,evidence_match,chapter,section,pdf_pages,preview
0,1,8,0.016393,True,1,1.1,[37],A database management system (DBMS) is a compu...
1,2,195,0.016393,False,2,2.7,[86],2.2. Discuss the main categories of data model...
2,3,40,0.016129,False,1,1.4.1,[46],[[SECTION 1.4.1: Database Administrators]]\nIn...
3,4,196,0.016129,False,2,2.7,"[86, 87]",architectures?\n2.10. Discuss some types of da...
4,5,97,0.015873,False,1,1.9,[59],1.2. What four main types of actions involve d...


# Experiment 6: So sánh kết quả

**Hit@5:** có ít nhất một chunk chứa đủ bằng chứng đã gán trong top-5.
**MRR@5:** nghịch đảo hạng đầu tiên chứa bằng chứng; không thấy thì bằng 0.
Sparse không coi kết quả điểm 0 là một kết quả tìm kiếm.

Đây là 16 câu trên 8 chủ đề, chưa phải benchmark tổng quát. Một đoạn khác trả lời đúng
nhưng không chứa bằng chứng đã gán có thể bị tính là miss. Không dùng Recall khi gold chưa đầy đủ.

In [201]:
def evaluate(name):
    rows = []
    for index, query in enumerate(queries):
        order = np.argsort(-scores[name][index], kind="stable")
        if name in ["tfidf", "bge_sparse"]:
            order = order[scores[name][index, order] > 0]
        ranks = np.flatnonzero(gold[index, order[:TOP_K]]) + 1
        rows.append({
            "method": name, "topic": query["topic"],
            "language": query["language"], "query": query["text"],
            "hit@5": int(len(ranks) > 0),
            "mrr@5": float(1 / ranks[0]) if len(ranks) else 0.0,
        })
    return pd.DataFrame(rows)

In [202]:
results = pd.concat([evaluate(name) for name in scores], ignore_index=True)
summary = results.groupby(["method", "language"])[["hit@5", "mrr@5"]].mean()
summary.round(3)

hit@5  mrr@5
method     language              
bge_dense  en        1.000  0.823
           vi        0.875  0.719
bge_multi  en        1.000  0.844
           vi        0.875  0.812
bge_sparse en        1.000  0.775
           vi        0.000  0.000
e5         en        1.000  0.938
           vi        0.625  0.500
hybrid     en        1.000  0.844
           vi        0.750  0.448
static     en        1.000  0.729
           vi        0.000  0.000
tfidf      en        0.875  0.708
           vi        0.000  0.000

## Xem câu nào chưa tìm được bằng chứng

In [203]:
results.loc[results["hit@5"] == 0, ["method", "language", "topic", "query"]]

,method,language,topic,query
1,tfidf,vi,dbms,Hệ quản trị cơ sở dữ liệu là gì?
2,tfidf,en,schema,What is a database schema?
3,tfidf,vi,schema,Lược đồ cơ sở dữ liệu là gì?
5,tfidf,vi,state,Dữ liệu trong cơ sở dữ liệu tại một thời điểm ...
7,tfidf,vi,logical,Loại độc lập dữ liệu nào cho phép thay đổi cấu...
9,tfidf,vi,physical,Vì sao có thể tổ chức lại các tệp vật lý mà kh...
11,tfidf,vi,multivalued,Một thuộc tính có thể có nhiều giá trị cho cùn...
13,tfidf,vi,derived,Vì sao tuổi có thể tính từ ngày sinh thay vì l...
15,tfidf,vi,key,Thuộc tính nào có thể xác định duy nhất từng t...
17,static,vi,dbms,Hệ quản trị cơ sở dữ liệu là gì?


## Đối chiếu top-5 của cùng một câu hỏi

In [204]:
QUERY_INDEX = 1
pd.concat(
    {name: top_results(name, QUERY_INDEX) for name in scores},
    names=["method", "row"],
)[["rank", "row_index", "evidence_match", "section", "pdf_pages", "preview"]]

rank  row_index evidence_match  section pdf_pages  \
method     row                                                      
static     0     1.0      213.0          False      3.2  [93, 94]   
           1     2.0      205.0          False  3.intro      [91]   
           2     3.0      189.0          False      2.6      [84]   
           3     4.0        0.0          False  1.intro      [35]   
           4     5.0      158.0          False    2.4.3      [77]   
e5         0     1.0        8.0           True      1.1      [37]   
           1     2.0      146.0          False    2.4.1  [73, 74]   
           2     3.0      202.0          False      2.7      [90]   
           3     4.0      130.0          False    2.3.1      [70]   
           4     5.0       40.0          False    1.4.1      [46]   
bge_dense  0     1.0        8.0           True      1.1      [37]   
           1     2.0       40.0          False    1.4.1      [46]   
           2     3.0      152.0          False    2.4.1      [75]   
           3     4.0      145.0          False    2.4.1      [73]   
           4     5.0       23.0          False    1.3.1  [41, 42]   
bge_sparse 0     1.0      195.0          False      2.7      [86]   
           1     2.0      196.0          False      2.7  [86, 87]   
           2     3.0       97.0          False      1.9      [59]   
bge_multi  0     1.0        8.0           True      1.1      [37]   
           1     2.0       40.0          False    1.4.1      [46]   
           2     3.0        7.0          False      1.1  [36, 37]   
           3     4.0       10.0          False      1.1      [37]   
           4     5.0      145.0          False    2.4.1      [73]   
hybrid     0     1.0        8.0           True      1.1      [37]   
           1     2.0      195.0          False      2.7      [86]   
           2     3.0       40.0          False    1.4.1      [46]   
           3     4.0      196.0          False      2.7  [86, 87]   
           4     5.0       97.0          False      1.9      [59]   

                                                          preview  
method     row                                                     
static     0    [[SECTION 3.2: A Sample Database Application]]...  
           1    . Section 3.8 introduces some UML class diagra...  
           2    The hierarchical model represents data as hier...  
           3    [[SECTION 1.intro: Chapter Introduction]]\nsto...  
           4    Application development environments, such as ...  
e5         0    A database management system (DBMS) is a compu...  
           1    Let us consider the top part of Figure 2.3 fir...  
           2    In this chapter, we follow the traditional app...  
           3    In DBMSs where a clear separation is maintaine...  
           4    [[SECTION 1.4.1: Database Administrators]]\nIn...  
bge_dense  0    A database management system (DBMS) is a compu...  
           1    [[SECTION 1.4.1: Database Administrators]]\nIn...  
           2    It is common to have the client program that a...  
           3    [[SECTION 2.4.1: DBMS Component Modules]]\nFig...  
           4    [[SECTION 1.3.1: Self-Describing Nature of a D...  
bge_sparse 0    2.2. Discuss the main categories of data model...  
           1    architectures?\n2.10. Discuss some types of da...  
           2    1.2. What four main types of actions involve d...  
bge_multi  0    A database management system (DBMS) is a compu...  
           1    [[SECTION 1.4.1: Database Administrators]]\nIn...  
           2    An example of a large commercial database is A...  
           3    It is not absolutely necessary to use general-...  
           4    [[SECTION 2.4.1: DBMS Component Modules]]\nFig...  
hybrid     0    A database management system (DBMS) is a compu...  
           1    2.2. Discuss the main categories of data model...  
           2    [[SECTION 1.4.1: Database Administrators]]\nIn...  
           3    architectures?\n2.10. Dis

## Đọc toàn bộ chunk và record nguồn của một kết quả

In [205]:
METHOD = "e5"
RANK = 1
result = top_results(METHOD, QUERY_INDEX).iloc[RANK - 1]
texts[indices.index(int(result["row_index"]))]

'A database management system (DBMS) is a computerized system that enables users to create and maintain a database. The DBMS is a general-purpose software system that facilitates the processes of defining, constructing, manipulating, and sharing databases among various users and applications. Defining a database involves specifying the data types, structures, and constraints of the data to be stored in the database. The database definition or descriptive information is also stored by the DBMS in the form of a database catalog or dictionary; it is called meta-data. Constructing the database is the process of storing the data on some storage medium that is controlled by the DBMS. Manipulating a database includes functions such as querying the database to retrieve specific data, updating the database to reflect changes in the miniworld, and generating reports from the data. Sharing a database allows multiple users and programs to access the database simultaneously.'

In [206]:
retrieved_chunk = chunks[int(result["row_index"])]
pd.DataFrame([
    {"record_id": source["record_id"], "pdf_page": source["pdf_page"] + 1,
     "text": records_by_id[source["record_id"]]["text"]}
    for source in retrieved_chunk.metadata["sources"]
])

,record_id,pdf_page,text
0,863264e73f7a57ea:p36:b2,37,A database management system (DBMS) is a compu...


## Thời gian encode

Đo một lượt trên máy hiện tại, không gồm download. `queries_s` là encode cả 16 câu.
`bge_all_three` tạo cả ba biểu diễn cùng lúc; không suy ra thời gian encode riêng từng nhánh.
Số thread=4, float32; E5/BGE batch=2, static batch=32. Đây là cấu hình thực nghiệm, chưa tối ưu.

In [207]:
timing = pd.DataFrame(times)
timing["documents_per_second"] = len(texts) / timing["documents_s"]
timing["query_encode_ms"] = timing["queries_s"] * 1000 / len(queries)
timing.round(3)

,method,load_s,documents_s,queries_s,documents_per_second,query_encode_ms
0,tfidf,0.000,0.127,0.004,1836.906,0.267
1,static,0.137,0.435,0.014,535.599,0.884
2,e5,5.368,33.123,0.397,7.034,24.823
3,bge_all_three,5.034,332.614,4.099,0.701,256.177


## Thời gian tính điểm trực tiếp

Dense/TF-IDF dùng phép nhân ma trận; sparse/multi-vector dùng vòng lặp Python đơn giản.
Đây là thời gian scoring trên corpus nhỏ, không phải benchmark backend hay ANN.
Hybrid cần cả dense+sparse scoring, rồi mới fusion.

In [208]:
scoring = pd.DataFrame([
    {"method": name, "scoring_ms_per_query": seconds * 1000 / len(queries)}
    for name, seconds in search_times.items()
])
scoring.loc[scoring["method"] == "hybrid", "scoring_ms_per_query"] += (
    search_times["bge_dense"] + search_times["bge_sparse"]
) * 1000 / len(queries)
scoring.round(3)

,method,scoring_ms_per_query
0,tfidf,0.469
1,static,1.960
2,e5,0.944
3,bge_dense,1.263
4,bge_sparse,1.851
5,bge_multi,2413.711
6,hybrid,3.436


## Dung lượng biểu diễn document

`numeric_mib` chỉ gồm vector và chỉ số số học, không gồm model, metadata, index hoặc overhead Python.
BGE sparse ước tính int32 token ID + float32 weight (8 byte/mục), không đo kích thước dict Python.
Hybrid dùng lại dense+sparse, không tạo vector mới.

In [209]:
size_summary = pd.DataFrame(sizes)
size_summary.round(3)

,method,representation,dimensions,nonzero_values,numeric_mib
0,tfidf,CSR sparse,16037,37820.0,0.289
1,static,one dense vector,1024,NaN,0.910
2,e5,one dense vector,384,NaN,0.341
3,bge_dense,one dense vector,1024,NaN,0.910
4,bge_sparse,token IDs + weights,250002,17014.0,0.130
5,bge_multi,token vectors,1024,NaN,143.996


## Lưu bảng kết quả để đối chiếu các lần chạy

In [210]:
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

In [211]:
results.to_csv(OUTPUT_PATH / "per_query.csv", index=False, encoding="utf-8-sig")

In [212]:
summary.to_csv(OUTPUT_PATH / "summary.csv", encoding="utf-8-sig")

In [213]:
timing.to_csv(OUTPUT_PATH / "timing.csv", index=False, encoding="utf-8-sig")

In [214]:
scoring.to_csv(OUTPUT_PATH / "scoring.csv", index=False, encoding="utf-8-sig")

In [215]:
size_summary.to_csv(OUTPUT_PATH / "representation.csv", index=False, encoding="utf-8-sig")

In [216]:
top_tables = pd.concat([
    top_results(name, index).assign(method=name, query_index=index,
                                    query=query["text"], language=query["language"])
    for name in scores
    for index, query in enumerate(queries)
], ignore_index=True)
top_tables.to_csv(OUTPUT_PATH / "top5.csv", index=False, encoding="utf-8-sig")

## Ghi lại input, model và môi trường của lần chạy

In [217]:
from importlib.metadata import version

manifest = {
    "strategy": STRATEGY, "chunks": len(texts), "queries": len(queries),
    "input_sha256": hashlib.sha256(CHUNK_PATH.read_bytes()).hexdigest(),
    "parser_sha256": hashlib.sha256(PARSER_PATH.read_bytes()).hexdigest(),
    "e5_revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3",
    "models": model_info,
    "torch_threads": 4, "batch_size": BATCH_SIZE, "rrf_k": 60, "rrf_candidates": 20,
    "packages": {name: version(name) for name in [
        "torch", "sentence-transformers", "FlagEmbedding", "numpy", "pandas", "scikit-learn"]},
    "cases": cases,
}

In [218]:
(OUTPUT_PATH / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

4707

# Nhận xét lượt chạy đã kiểm tra — 06/10/2026

Lượt này dùng **233 chunk có nguồn của structure_aware_recursive**, 8 cặp câu Anh–Việt.
Các số dưới là snapshot của lượt mặc định. Nếu đổi STRATEGY hoặc bộ câu hỏi, đọc bảng tính lại ở trên.

- **Tiếng Anh:** TF-IDF tìm được bằng chứng cho 7/8 câu; các phương pháp còn lại 8/8.
  E5 có MRR@5 cao nhất trong lượt này: 0.938. Hit@5 bằng nhau không có nghĩa thứ hạng bằng nhau.
- **Tiếng Việt:** E5 đạt 5/8; BGE dense và multi-vector cùng đạt 7/8.
  MRR@5 của BGE multi-vector là 0.812, của BGE dense là 0.719.
  TF-IDF, static tiếng Anh và BGE sparse đều đạt 0/8 trên bằng chứng đã gán.
- **Hybrid:** đạt 6/8 câu Việt, thấp hơn BGE dense ở lượt này. Nhánh lexical không có cùng
  khả năng tìm xuyên ngôn ngữ như dense; kết hợp thứ hạng vẫn cần đánh giá trên query thực tế.
- **Chi phí biểu diễn:** BGE multi-vector dùng khoảng 144 MiB dữ liệu số cho document,
  BGE dense khoảng 0.91 MiB. Multi-vector giữ chi tiết hơn nhưng lưu nhiều vector hơn.
- **Encode CPU:** static khoảng 0.07 giây, E5 khoảng 13.03 giây,
  BGE-M3 tạo cả ba output khoảng 229.13 giây cho corpus này. Không dùng thời gian joint encode
  để kết luận thời gian riêng của BGE dense, sparse hay multi-vector.

Đây là kết quả truy hồi **đúng khoảng bằng chứng đã gán**, không phải đo mọi câu trả lời hợp lệ.
Chưa đủ để chọn model tốt nhất cho toàn sách. Với câu miss, xem top-5 và record nguồn để phân biệt
đoạn không trả lời được với đoạn trả lời được nhưng nằm ngoài gold hiện tại.


# Cách đọc kết quả và thực nghiệm tiếp

- TF-IDF giữ từ/ngram; static giữ ngữ nghĩa token nhưng không xử lý ngữ cảnh;
  dense nén đoạn thành một vector; sparse neural giữ trọng số token; multi-vector giữ nhiều vector.
- So sánh BGE dense/sparse/multi trên cùng backbone. So E5 với BGE dense là đổi model trong cùng nhóm.
- Đọc riêng Anh/Việt; static tiếng Anh không là đối chứng công bằng cho năng lực đa ngôn ngữ.
- Không so độ lớn score giữa các phương pháp. Xem bằng chứng đúng và thứ hạng.
- Bộ câu hỏi nhỏ, gold chưa đầy đủ, thời gian chỉ một lượt; không kết luận nhóm nào luôn tốt nhất.
- Muốn thử output Chunker khác: đổi `STRATEGY` thành `strict_fixed` hoặc `global_recursive`, rồi Run All.
  Notebook sẽ kiểm tra lại coverage bằng chứng và lưu bảng vào thư mục chiến lược tương ứng.
  Lượt mặc định này giữ chunking cố định để quan sát embedding.

Nguồn: [TF-IDF](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction),
[StaticEmbedding](https://huggingface.co/sentence-transformers/static-retrieval-mrl-en-v1),
[E5](https://huggingface.co/intfloat/multilingual-e5-small),
[BGE-M3](https://huggingface.co/BAAI/bge-m3),
[FlagEmbedding scoring](https://github.com/FlagOpen/FlagEmbedding/blob/master/FlagEmbedding/inference/embedder/encoder_only/m3.py).

# Phụ lục: tải model nếu cache chưa có

Hai cell dưới **không tự chạy trong Run All**. Bỏ comment rồi chạy riêng khi cần tải trên máy mới.
Tải model công khai từ Hugging Face; không upload nội dung sách. Sau khi tải xong, giữ comment và Run All.
Model revision nằm trong `model_info` ở Setup và manifest để biết đúng phiên bản của thực nghiệm.

In [219]:
# from huggingface_hub import snapshot_download
# snapshot_download("sentence-transformers/static-retrieval-mrl-en-v1",
#                   local_dir=CACHE_PATH / "static",
#                   revision=model_info["static"]["revision"],
#                   allow_patterns=["*.json", "*.safetensors", "0_StaticEmbedding/*"])

In [220]:
# from huggingface_hub import snapshot_download
# snapshot_download("BAAI/bge-m3", local_dir=CACHE_PATH / "bge_m3",
#                   revision=model_info["bge_m3"]["revision"],
#                   allow_patterns=["*.json", "pytorch_model.bin", "*.pt", "*.model"])